# HNSW tự viết, so với IVF và FAISS trên embedding Zalo Legal

Bạn cài **HNSW** (Hierarchical Navigable Small World, Malkov & Yashunin 2016), index ANN mặc định của Qdrant,
pgvector, Chroma, Weaviate, rồi đo nó trên vector `bge-m3` thật của corpus Zalo Legal: recall của ANN so với kNN
chính xác, số phép tính khoảng cách, thời gian mỗi query, và so với IVF của bài Basic 02 và với FAISS.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama đang chạy với `bge-m3`. Lần đầu notebook embed cả corpus (61 425 điều) và cache ở
  `data/raw/zalo-legal/cache/bge-m3.npy`; module 13 dùng lại đúng tệp này.
- Chấm: `uv run pytest tracks/ai-engineer/modules/12-embeddings-vector-db/exercises/intermediate/01-hnsw-from-scratch -q`

In [ ]:
# explore: tìm gốc repo, tải Zalo Legal nếu chưa có (khoảng 110 MB, chỉ lần đầu)
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ZALO = ensure("zalo-legal")
CACHE = ZALO / "cache"
CACHE.mkdir(exist_ok=True)
print(ZALO, sorted(p.name for p in ZALO.iterdir()))

In [ ]:
import heapq
import math
import time

import numpy as np

## 1. Tầng của một nút

Mỗi nút mới được gán một tầng cao nhất $$\ell$$ theo phân phối mũ rời rạc:

$$
\ell = \left\lfloor -\ln(1 - u)\cdot m_L \right\rfloor, \qquad u \sim \mathcal{U}[0, 1), \qquad m_L = \frac{1}{\ln M}
$$

Với $$m_L = 1/\ln M$$, xác suất một nút có mặt ở tầng $$\ge 1$$ là $$1/M$$, ở tầng $$\ge 2$$ là $$1/M^2$$...: mỗi tầng trên
thưa hơn tầng dưới $$M$$ lần, giống skip list.

`random_level(rng, m_l) -> int`: lấy **đúng một** số `u = rng.random()` rồi áp công thức trên (dùng `1 - u` để
không bao giờ tính `ln 0`).

In [ ]:
def random_level(rng: np.random.Generator, m_l: float) -> int:
    """Tầng cao nhất của nút mới: int(-ln(1 - u) * m_l) với u = rng.random()."""
    # TODO
    raise NotImplementedError

## 2. Lớp `HNSW`

Quy ước (đã có sẵn trong `__init__`, `max_degree`, `dist`, `add`):

- Vector đã chuẩn hoá L2; khoảng cách là **cosine distance** $$1 - q^\top x$$ (`dist(q, ids)` trả mảng khoảng cách
  từ `q` tới các vector `ids`, và đếm số phép tính vào `n_dist`). Mọi phép tính khoảng cách phải đi qua `dist`.
- `graph[layer]` là `dict` từ id nút tới **list** id láng giềng ở tầng đó; `levels[i]` là tầng cao nhất của nút `i`;
  `entry_point` là nút có tầng cao nhất (nút đầu tiên đạt tầng đó).
- Bậc tối đa: `M` ở tầng trên, `2 * M` ở tầng 0 (`max_degree`).

Bốn phương thức cần viết:

1. `search_layer(q, entry_points, ef, layer) -> list[(khoảng cách, id)]`: beam search trên **một** tầng (Algorithm 2
   của bài báo). Giữ hai tập: `candidates` (min-heap theo khoảng cách, nút còn phải mở rộng) và `best` (tối đa `ef`
   nút gần nhất đã thấy). Lặp: lấy ứng viên gần nhất; nếu nó xa hơn phần tử xa nhất của `best` thì dừng; ngược lại
   xét các láng giềng **chưa thăm**, láng giềng nào gần hơn phần tử xa nhất của `best` (hoặc `best` chưa đủ `ef`) thì
   vào cả hai tập (cắt `best` về `ef`). Trả `best` sắp tăng dần theo khoảng cách (bằng nhau thì id nhỏ trước).
   Điểm vào được tính là đã thăm và có mặt trong kết quả nếu đủ gần.
2. `select_neighbors(candidates, m) -> list[int]`: `candidates` là các cặp (khoảng cách tới nút gốc, id) **đã sắp
   tăng dần**. Nếu `heuristic=False`: `m` id đầu. Nếu `heuristic=True` (Algorithm 4, bản rút gọn): duyệt lần lượt, giữ
   ứng viên `e` khi nó **gần nút gốc hơn** gần mọi láng giềng đã giữ (`d(gốc, e) < d(e, r)` với mọi `r` đã chọn), dừng
   khi đủ `m`. Không bù thêm ứng viên bị loại.
3. `_insert(v)`: thêm một vector (Algorithm 1). Id mới là `len(vectors)` trước khi thêm. Tầng `level = random_level(self.rng,
   self.m_l)`; đăng ký nút ở mọi tầng `0..level` (`graph[lc][new] = []`, thêm tầng mới vào `graph` nếu cần). Nút
   đầu tiên chỉ trở thành `entry_point`. Các nút sau:
   - Từ tầng của `entry_point` xuống tầng `level + 1`: tìm tham lam `search_layer(v, ep, 1, lc)`, điểm vào tầng dưới
     là nút gần nhất tìm được.
   - Từ tầng `min(level, tầng của entry_point)` xuống 0: `found = search_layer(v, ep, ef_construction, lc)`; láng
     giềng của nút mới là `select_neighbors(found, M)`; nối **hai chiều**; láng giềng nào vượt `max_degree(lc)` thì tỉa
     lại bằng `select_neighbors` trên danh sách của nó (sắp theo khoảng cách tới **nó**, bằng nhau giữ thứ tự cũ);
     điểm vào tầng dưới là mọi id trong `found`.
   - Nếu `level` lớn hơn tầng của `entry_point` thì nút mới thành `entry_point`.
4. `search(q, k, ef=50) -> list[int]`: tham lam với `ef = 1` từ tầng cao nhất xuống tầng 1, rồi
   `search_layer(q, ep, max(ef, k), 0)` và trả `k` id đầu.

In [ ]:
class HNSW:
    """HNSW tối giản cho vector đã chuẩn hoá L2; khoảng cách = 1 - tích vô hướng."""

    def __init__(self, dim: int, M: int = 16, ef_construction: int = 100, seed: int = 0, heuristic: bool = True):
        self.dim, self.M, self.ef_construction, self.heuristic = dim, M, ef_construction, heuristic
        self.m_l = 1.0 / math.log(M)
        self.rng = np.random.default_rng(seed)
        self.vectors = np.zeros((0, dim), dtype=np.float32)
        self.levels: list[int] = []
        self.graph: list[dict[int, list[int]]] = []
        self.entry_point: int | None = None
        self.n_dist = 0

    def max_degree(self, layer: int) -> int:
        return 2 * self.M if layer == 0 else self.M

    def dist(self, q: np.ndarray, ids) -> np.ndarray:
        """Cosine distance từ q tới các vector ids; cộng số phép tính vào n_dist."""
        ids = np.asarray(ids, dtype=np.int64)
        self.n_dist += len(ids)
        return 1.0 - self.vectors[ids] @ q

    def add(self, x: np.ndarray) -> None:
        for v in np.asarray(x, dtype=np.float32):
            self._insert(v)

    def search_layer(self, q: np.ndarray, entry_points: list[int], ef: int, layer: int) -> list[tuple[float, int]]:
        """Beam search trên một tầng; tối đa ef cặp (khoảng cách, id) tăng dần."""
        # TODO
        raise NotImplementedError

    def select_neighbors(self, candidates: list[tuple[float, int]], m: int) -> list[int]:
        """Chọn tối đa m láng giềng từ các cặp (khoảng cách tới nút gốc, id) đã sắp tăng dần."""
        # TODO
        raise NotImplementedError

    def _insert(self, v: np.ndarray) -> None:
        """Thêm một vector đã chuẩn hoá (Algorithm 1)."""
        # TODO
        raise NotImplementedError

    def search(self, q: np.ndarray, k: int, ef: int = 50) -> list[int]:
        """k id gần q nhất: tham lam ở các tầng trên, beam search max(ef, k) ở tầng 0."""
        # TODO
        raise NotImplementedError

In [ ]:
# explore: kiểm nhanh trên dữ liệu sinh: 20 cụm x 100 điểm, 32 chiều; heuristic so với chọn m gần nhất
rng = np.random.default_rng(0)
centers = rng.normal(size=(20, 32))
toy = np.concatenate([c + 0.5 * rng.normal(size=(100, 32)) for c in centers])
toy = (toy / np.linalg.norm(toy, axis=1, keepdims=True))[rng.permutation(2000)].astype(np.float32)
toy_q = toy[rng.choice(2000, 50)] + 0.05 * rng.normal(size=(50, 32))
toy_q = (toy_q / np.linalg.norm(toy_q, axis=1, keepdims=True)).astype(np.float32)
toy_exact = np.argsort(-(toy_q @ toy.T), axis=1, kind="stable")[:, :10]
for heuristic in (True, False):
    h = HNSW(32, M=8, ef_construction=40, seed=0, heuristic=heuristic)
    h.add(toy)
    for ef in (10, 40):
        r = np.mean([len(set(h.search(q, 10, ef)) & set(e)) / 10 for q, e in zip(toy_q, toy_exact)])
        print(f"heuristic={heuristic!s:5}  ef={ef:3d}  recall ANN@10 = {r:.3f}")

## 3. IVF của bài Basic 02 (dán vào)

Notebook tự chứa: **dán** `assign`, `kmeans` và `IVFIndex` từ lời giải Basic 02 của bạn vào cell dưới, giữ nguyên chữ
ký (hàm `sq_dists` đã có sẵn). Một test nhỏ kiểm lại chúng để bảng so sánh ở cuối bài đáng tin.

In [ ]:
def sq_dists(x: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """Bình phương khoảng cách Euclid (n, n_centroids), không âm."""
    x = np.asarray(x, dtype=np.float32)
    c = np.asarray(centroids, dtype=np.float32)
    d = (x * x).sum(1)[:, None] + (c * c).sum(1)[None, :] - 2 * (x @ c.T)
    return np.maximum(d, 0)


def assign(x: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """Chỉ số centroid gần nhất (Basic 02)."""
    # TODO
    raise NotImplementedError


def kmeans(x: np.ndarray, n_clusters: int, n_iter: int = 20, seed: int = 0) -> np.ndarray:
    """Lloyd k-means, khởi tạo bằng rng.choice không lặp (Basic 02)."""
    # TODO
    raise NotImplementedError


class IVFIndex:
    def __init__(self, nlist: int, n_iter: int = 20, seed: int = 0):
        self.nlist, self.n_iter, self.seed = nlist, n_iter, seed
        self.centroids: np.ndarray | None = None
        self.lists: list[np.ndarray] = []
        self.vectors = np.zeros((0, 0), dtype=np.float32)

    def train(self, x: np.ndarray) -> None:
        """Basic 02."""
        # TODO
        raise NotImplementedError

    def add(self, x: np.ndarray) -> None:
        """Basic 02."""
        # TODO
        raise NotImplementedError

    def search(self, q: np.ndarray, k: int, nprobe: int = 1) -> tuple[list[int], int]:
        """Basic 02."""
        # TODO
        raise NotImplementedError

## 4. Vector thật: embed corpus một lần, cache cho cả module 12 và 13

Cell dưới đọc cache nếu đã có (và danh sách id khớp corpus), nếu không thì embed `title + "\n" + text` của 61 425
điều qua `llmkit.embed` theo lô 32, chuẩn hoá L2 và ghi `bge-m3.npy` + `bge-m3.ids.json`. Lần đầu mất khoảng 45 phút
trên laptop GPU 4GB (Ollama chỉ chạy CPU thì lâu hơn nhiều); tệp không bao giờ bị ghi đè nếu đã tồn tại.

In [ ]:
# explore: embed corpus (hoặc đọc cache) và 300 câu hỏi ghim
import json

import llmkit

corpus = [json.loads(line) for line in (ZALO / "corpus.jsonl").open(encoding="utf-8")]
uids = [d["_id"] for d in corpus]


def safe_embed(texts: list[str]) -> np.ndarray:
    """llmkit.embed; văn bản làm bge-m3 lỗi (NaN, HTTP 500) được thử sửa nhẹ, cuối cùng thành vector 0."""
    try:
        return llmkit.embed(texts)
    except llmkit.LLMError:
        rows = []
        for text in texts:
            for variant in (text, text.strip() + " .", text[: len(text) // 2]):
                try:
                    rows.append(llmkit.embed([variant])[0])
                    break
                except llmkit.LLMError:
                    continue
            else:
                print("không embed được:", text[:80])
                rows.append(np.zeros(1024, np.float32))
        return np.stack(rows)


matrix_path, ids_path = CACHE / "bge-m3.npy", CACHE / "bge-m3.ids.json"
if matrix_path.exists() and ids_path.exists() and json.loads(ids_path.read_text(encoding="utf-8")) == uids:
    matrix = np.load(matrix_path)
    print("đọc cache", matrix_path, matrix.shape)
elif matrix_path.exists():
    raise RuntimeError(f"{matrix_path} có sẵn nhưng id không khớp corpus: kiểm tra lại, không ghi đè")
else:
    texts = [d["title"] + "\n" + d["text"] for d in corpus]
    t0 = time.perf_counter()
    raw = np.concatenate([safe_embed(texts[i:i + 32]) for i in range(0, len(texts), 32)]).astype(np.float32)
    norms = np.linalg.norm(raw, axis=1, keepdims=True)
    matrix = raw / np.where(norms == 0, 1, norms)
    print(f"embed corpus: {(time.perf_counter() - t0) / 60:.1f} phút")
    np.save(matrix_path, matrix)
    ids_path.write_text(json.dumps(uids, ensure_ascii=False), encoding="utf-8")

qids = []
for line in (ZALO / "qrels" / "test.jsonl").open(encoding="utf-8"):
    qid = json.loads(line)["query-id"]
    if qid not in qids:
        qids.append(qid)
qids = qids[:300]
qtext = {q["_id"]: q["text"] for q in map(json.loads, (ZALO / "queries.jsonl").open(encoding="utf-8"))}
queries = llmkit.embed([qtext[q] for q in qids])
queries = queries / np.linalg.norm(queries, axis=1, keepdims=True)
print("ma trận", matrix.shape, matrix.dtype, "| câu hỏi", queries.shape)

## 5. Đo: exact, IVF và HNSW tự viết trên 20 000 điều đầu

HNSW bằng Python thuần chậm khi dựng (mỗi nút mới vài trăm phép tính khoảng cách cộng chi phí của heap và `dict`),
nên phần tự viết chạy trên **20 000 điều đầu** của corpus; FAISS ở mục sau chạy cả hai cỡ. Ground truth là kNN chính
xác bằng một phép nhân ma trận. Query là 300 câu hỏi ghim (dù điều đúng của chúng có thể không nằm trong 20 000 điều
này: recall của ANN chỉ so với kNN chính xác trên cùng tập).

In [ ]:
# explore: ground truth và HNSW tự viết (dựng mất vài phút)
base = np.ascontiguousarray(matrix[:20000])
t0 = time.perf_counter()
exact = np.argsort(-(queries @ base.T), axis=1, kind="stable")[:, :10]
exact_ms = 1000 * (time.perf_counter() - t0) / len(queries)
print(f"exact (numpy, cả lô 300 câu): {exact_ms:.2f} ms/câu, mỗi câu {len(base)} phép tính khoảng cách")


def ann_recall(approx, truth, k=10):
    return float(np.mean([len(set(list(a)[:k]) & set(list(t)[:k])) / k for a, t in zip(approx, truth)]))


t0 = time.perf_counter()
hnsw = HNSW(1024, M=16, ef_construction=100, seed=0)
hnsw.add(base)
hnsw_build = time.perf_counter() - t0
deg0 = np.mean([len(v) for v in hnsw.graph[0].values()])
print(f"dựng HNSW tự viết: {hnsw_build:.0f} s, {len(hnsw.graph)} tầng, số nút mỗi tầng",
      [len(g) for g in hnsw.graph], f"| bậc trung bình tầng 0: {deg0:.1f}")

rows = []
for ef in (10, 20, 40, 80, 160):
    hnsw.n_dist = 0
    t0 = time.perf_counter()
    found = [hnsw.search(q, 10, ef) for q in queries]
    ms = 1000 * (time.perf_counter() - t0) / len(queries)
    rows.append(("HNSW tự viết", f"ef={ef}", ann_recall(found, exact), ms, hnsw.n_dist / len(queries)))

In [ ]:
# explore: IVF tự viết, nlist = 128 (khoảng căn bậc hai của 20 000)
t0 = time.perf_counter()
ivf = IVFIndex(128, n_iter=10, seed=0)
ivf.train(base)
ivf.add(base)
ivf_build = time.perf_counter() - t0
sizes = np.array([len(l) for l in ivf.lists])
print(f"dựng IVF tự viết: {ivf_build:.1f} s | kích thước danh sách min/trung vị/max: {sizes.min()}/{int(np.median(sizes))}/{sizes.max()}")
for nprobe in (1, 2, 4, 8, 16, 32):
    t0 = time.perf_counter()
    out = [ivf.search(q, 10, nprobe) for q in queries]
    ms = 1000 * (time.perf_counter() - t0) / len(queries)
    rows.append(("IVF tự viết", f"nprobe={nprobe}", ann_recall([o[0] for o in out], exact), ms,
                 np.mean([o[1] for o in out])))

## 6. FAISS trên cùng dữ liệu, rồi trên cả corpus

`IndexHNSWFlat` và `IndexIVFFlat` của FAISS cài cùng thuật toán bằng C++ (SIMD, không có overhead của Python). Dùng
`METRIC_INNER_PRODUCT` vì vector đã chuẩn hoá. Đặt số luồng 4 để không chiếm máy.

FAISS được đo **cả lô** 300 câu trong một lời gọi `search` (cách dùng thường gặp khi chạy batch), còn code tự viết đo
từng câu trong vòng lặp Python: cột ms/câu của hai nhóm không so trực tiếp được. Cột recall thì so được.

In [ ]:
# explore: FAISS HNSW và IVF trên 20 000 điều
import faiss

faiss.omp_set_num_threads(4)


def faiss_rows(name, index, x, truth, params, setter):
    out = []
    for p in params:
        setter(index, p)
        t0 = time.perf_counter()
        _, ids = index.search(queries, 10)
        ms = 1000 * (time.perf_counter() - t0) / len(queries)
        out.append((name, p if isinstance(p, str) else f"{p}", ann_recall(ids, truth), ms, float("nan")))
    return out


def build_faiss_hnsw(x):
    index = faiss.IndexHNSWFlat(x.shape[1], 16, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 100
    t0 = time.perf_counter()
    index.add(x)
    return index, time.perf_counter() - t0


def build_faiss_ivf(x, nlist):
    quantizer = faiss.IndexFlatIP(x.shape[1])
    index = faiss.IndexIVFFlat(quantizer, x.shape[1], nlist, faiss.METRIC_INNER_PRODUCT)
    t0 = time.perf_counter()
    index.train(x)
    index.add(x)
    return index, time.perf_counter() - t0


fh, fh_build = build_faiss_hnsw(base)
rows += faiss_rows("FAISS HNSW", fh, base, exact, [10, 20, 40, 80, 160],
                   lambda ix, p: setattr(ix.hnsw, "efSearch", p))
fi, fi_build = build_faiss_ivf(base, 128)
rows += faiss_rows("FAISS IVF", fi, base, exact, [1, 2, 4, 8, 16, 32], lambda ix, p: setattr(ix, "nprobe", p))
print(f"dựng FAISS HNSW: {fh_build:.1f} s | FAISS IVF (train + add): {fi_build:.1f} s | HNSW tự viết: {hnsw_build:.0f} s")

import pandas as pd

table = pd.DataFrame(rows, columns=["index", "tham số", "recall ANN@10", "ms/câu", "số khoảng cách/câu"])
print(table.round({"recall ANN@10": 3, "ms/câu": 3, "số khoảng cách/câu": 0}).to_string(index=False))

In [ ]:
# explore: FAISS trên cả 61 425 điều: recall và độ trễ khi corpus lớn gấp 3
full = np.ascontiguousarray(matrix)
t0 = time.perf_counter()
exact_full = np.argsort(-(queries @ full.T), axis=1, kind="stable")[:, :10]
print(f"exact cả corpus (numpy, cả lô): {1000 * (time.perf_counter() - t0) / len(queries):.2f} ms/câu")
fh_full, b1 = build_faiss_hnsw(full)
fi_full, b2 = build_faiss_ivf(full, 256)
big = faiss_rows("FAISS HNSW 61k", fh_full, full, exact_full, [20, 40, 80, 160], lambda ix, p: setattr(ix.hnsw, "efSearch", p))
big += faiss_rows("FAISS IVF 61k (nlist 256)", fi_full, full, exact_full, [4, 8, 16, 32], lambda ix, p: setattr(ix, "nprobe", p))
print(f"dựng HNSW 61k: {b1:.1f} s | IVF 61k: {b2:.1f} s")
print(pd.DataFrame(big, columns=["index", "tham số", "recall ANN@10", "ms/câu", "_"]).drop(columns="_").round(3).to_string(index=False))

In [ ]:
# explore: chọn láng giềng heuristic so với m gần nhất trên vector thật (5 000 điều đầu, cho nhanh)
small = base[:5000]
small_exact = np.argsort(-(queries @ small.T), axis=1, kind="stable")[:, :10]
for heuristic in (True, False):
    h = HNSW(1024, M=16, ef_construction=100, seed=0, heuristic=heuristic)
    h.add(small)
    for ef in (10, 40):
        r = ann_recall([h.search(q, 10, ef) for q in queries], small_exact)
        print(f"5 000 điều, heuristic={heuristic!s:5} ef={ef:3d}: recall ANN@10 = {r:.3f}")

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Ở cùng recall khoảng 0.95, HNSW và IVF tự viết mỗi cái tính bao nhiêu khoảng cách mỗi câu, so với 20 000 của kNN
  chính xác? Vì sao "số khoảng cách" là thước đo công bằng hơn "ms/câu" khi so code Python với C++?
- HNSW tự viết và FAISS HNSW cùng tham số `M`, `efConstruction`, `efSearch`: recall có giống nhau không, thời gian
  chênh bao nhiêu lần, và phần chênh đó đến từ đâu?
- Trên dữ liệu có cụm, bỏ heuristic chọn láng giềng thì chuyện gì xảy ra với đồ thị, và vì sao tăng `ef` không cứu được?

Sau đó làm phần trắc nghiệm của bài trong app.